In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from src.scoutai_common import get_engine, load_master_view

engine = get_engine()
df = load_master_view(engine)
df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
df = df[df['current_market_value'] > 0].copy()

cluster_features = ['age', 'total_appearances', 'international_caps', 'log_market_value']
for col in cluster_features:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

X_cluster = df[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)
df['PCA1'] = pca_components[:, 0]
df['PCA2'] = pca_components[:, 1]
print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")

In [ ]:
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)
cluster_df = pd.DataFrame(cluster_centers, columns=cluster_features)
cluster_df['avg_market_value'] = np.expm1(cluster_df['log_market_value'])

def label_cluster(row):
    if row['avg_market_value'] > 40_000_000:
        return "Elite Superstars"
    elif row['age'] > 29:
        return "Experienced Veterans"
    elif row['age'] < 24 and row['total_appearances'] < 100:
        return "Developing Prospects"
    else:
        return "Prime / Core Squad"

cluster_df['Cluster_Name'] = cluster_df.apply(label_cluster, axis=1)

name_counts = cluster_df['Cluster_Name'].value_counts()
duplicate_names = name_counts[name_counts > 1].index
if len(duplicate_names) > 0:
    seen = {}
    new_names = []
    for name in cluster_df['Cluster_Name']:
        if name in duplicate_names:
            seen[name] = seen.get(name, 0) + 1
            new_names.append(f"{name} ({seen[name]})")
        else:
            new_names.append(name)
    cluster_df['Cluster_Name'] = new_names

cluster_mapping = cluster_df['Cluster_Name'].to_dict()
df['Player_Profile'] = df['Cluster'].map(cluster_mapping)

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='PCA1', y='PCA2', hue='Player_Profile', palette='tab10', data=df,
                 alpha=0.6, edgecolor='k', s=60)
plt.title("ScoutAI: Player Segmentation (K-Means Clustering)", fontsize=16, fontweight='bold')
plt.xlabel("Experience & Value Dimension (PCA1)", fontsize=12)
plt.ylabel("Age & International Caps Dimension (PCA2)", fontsize=12)
plt.legend(title='Machine Learned Profiles', fontsize=10, title_fontsize=12, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../images/kmeans_player_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

for idx, row in cluster_df.iterrows():
    n_players = (df['Cluster'] == idx).sum()
    print(f"Profile: {row['Cluster_Name']}  (n={n_players})")
    print(f"  Avg Age: {row['age']:.1f} | Avg Appearances: {row['total_appearances']:.0f} | "
          f"Avg Int. Caps: {row['international_caps']:.1f} | Avg Value: E{row['avg_market_value']:,.0f}\n")